In [1]:
import glob
import os
import xml.etree.ElementTree as ET
import pandas as pd

In [2]:
DATA_DIR = "/media/zenusha/Extra/Notes/Big_Data/Data_sets/18508_122062_2026-07-31_03-06-43_stagecoach-scmy-route-schedule-data-transxchange_2_4"
OUTPUT_DIR = "./processed_data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
stops_list = []
links_list = []
journeys_list = []

In [4]:
xml_files = glob.glob(os.path.join(DATA_DIR, "*.xml"))
print(f"Starting ingestion of {len(xml_files)} XML files...")

for idx, file_path in enumerate(xml_files):
    filename = os.path.basename(file_path)
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        # Handle XML Namespaces dynamically
        ns = (
            {"txc": root.tag.split("}")[0].strip("{")}
            if "}" in root.tag
            else {}
        )
        pfx = "txc:" if ns else ""

        # 1. Parse Annotated Stop Points (Nodes)
        for stop in root.findall(f".//{pfx}AnnotatedStopPointRef", ns):
            stop_ref = stop.findtext(f"{pfx}StopPointRef", "", ns)
            common_name = stop.findtext(f"{pfx}CommonName", "", ns)
            locality = stop.findtext(f"{pfx}LocalityName", "", ns)
            stops_list.append(
                {
                    "stop_ref": stop_ref,
                    "common_name": common_name,
                    "locality": locality,
                    "file_source": filename,
                }
            )

        # 2. Parse Route Links (Edges)
        for link in root.findall(f".//{pfx}RouteLink", ns):
            link_id = link.get("id")
            from_stop = link.findtext(
                f"{pfx}From/{pfx}StopPointRef", "", ns
            )
            to_stop = link.findtext(f"{pfx}To/{pfx}StopPointRef", "", ns)
            distance = link.findtext(f"{pfx}Distance", "0", ns)
            direction = link.findtext(f"{pfx}Direction", "unknown", ns)

            links_list.append(
                {
                    "link_id": link_id,
                    "from_stop": from_stop,
                    "to_stop": to_stop,
                    "distance_meters": float(distance)
                    if distance.isdigit()
                    else 0.0,
                    "direction": direction,
                    "file_source": filename,
                }
            )

        # 3. Parse Vehicle Journeys (Timetable Rows)
        for vj in root.findall(f".//{pfx}VehicleJourney", ns):
            vj_code = vj.findtext(f"{pfx}PrivateCode", "", ns)
            line_ref = vj.findtext(f"{pfx}LineRef", "", ns)
            operator_ref = vj.findtext(f"{pfx}OperatorRef", "", ns)
            dept_time = vj.findtext(
                f"{pfx}DepartureTime", "", ns
            )

            journeys_list.append(
                {
                    "journey_code": vj_code,
                    "line_ref": line_ref,
                    "operator_ref": operator_ref,
                    "departure_time": dept_time,
                    "file_source": filename,
                }
            )

    except Exception as e:
        print(f"Error parsing {filename}: {e}")

Starting ingestion of 123 XML files...


In [5]:
# Build DataFrames
df_stops = (
    pd.DataFrame(stops_list).drop_duplicates(subset=["stop_ref"]).reset_index(drop=True)
)
df_links = pd.DataFrame(links_list)
df_journeys = pd.DataFrame(journeys_list)

# Export processed files
df_stops.to_csv(os.path.join(OUTPUT_DIR, "bus_stops.csv"), index=False)
df_links.to_csv(os.path.join(OUTPUT_DIR, "route_links.csv"), index=False)
df_journeys.to_csv(os.path.join(OUTPUT_DIR, "schedules.csv"), index=False)

print("\n--- PHASE 1 COMPLETE ---")
print(f"Unique Bus Stops (Nodes): {len(df_stops):,}")
print(f"Total Route Links (Edges): {len(df_links):,}")
print(f"Total Vehicle Journeys:     {len(df_journeys):,}")


--- PHASE 1 COMPLETE ---
Unique Bus Stops (Nodes): 4,346
Total Route Links (Edges): 19,316
Total Vehicle Journeys:     6,866
